# NanoVision Net X — Image Reconstruction (Colab)

This notebook now uses the requested dataset:
`https://github.com/Charith003/nu_data.git`

Workflow:
- clone dataset repo in Colab
- load grayscale images from the cloned folder
- generate degraded/noisy inputs
- train a convolutional autoencoder for reconstruction
- evaluate with PSNR/SSIM and visualize improvements


In [ ]:
# Install dependencies (Colab-safe)
!pip -q install scikit-image tensorflow pillow


In [ ]:
# Clone requested dataset
!rm -rf nu_data
!git clone https://github.com/Charith003/nu_data.git

from pathlib import Path
root = Path("nu_data")
print("Cloned:", root.resolve())
print("Top-level entries:")
for p in sorted(root.iterdir())[:20]:
    print("-", p)


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

import tensorflow as tf
from tensorflow.keras import layers, models

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)


In [ ]:
# Load images recursively from cloned dataset
DATA_ROOT = "nu_data"
TARGET_SIZE = (128, 128)
MAX_IMAGES = 3000  # adjust based on Colab memory/time

valid_ext = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
all_files = []
for dirpath, _, filenames in os.walk(DATA_ROOT):
    for fn in filenames:
        ext = os.path.splitext(fn)[1].lower()
        if ext in valid_ext:
            all_files.append(os.path.join(dirpath, fn))

all_files = sorted(all_files)
if len(all_files) == 0:
    raise RuntimeError("No images found in nu_data. Please inspect repo structure.")

all_files = all_files[:MAX_IMAGES]
print(f"Found {len(all_files)} image files (using up to {MAX_IMAGES}).")

def load_gray(path, target_size=(128, 128)):
    img = Image.open(path).convert("L").resize(target_size)
    arr = np.asarray(img, dtype=np.float32) / 255.0
    return arr

images = np.stack([load_gray(p, TARGET_SIZE) for p in all_files], axis=0)
images = np.expand_dims(images, axis=-1)

# train/test split
perm = np.random.permutation(len(images))
images = images[perm]
split = int(0.8 * len(images))
x_train = images[:split]
x_test = images[split:]

print("x_train:", x_train.shape, "x_test:", x_test.shape)


In [ ]:
# Create degraded/noisy inputs to reconstruct
def degrade(x, noise_factor=0.25):
    noisy = x + noise_factor * np.random.normal(0, 1, size=x.shape)
    return np.clip(noisy, 0.0, 1.0)

x_train_noisy = degrade(x_train)
x_test_noisy = degrade(x_test)

def build_autoencoder(input_shape=(128, 128, 1)):
    inputs = layers.Input(shape=input_shape)

    # Encoder
    x = layers.Conv2D(32, 3, activation="relu", padding="same")(inputs)
    x = layers.MaxPooling2D(2, padding="same")(x)
    x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
    x = layers.MaxPooling2D(2, padding="same")(x)
    x = layers.Conv2D(128, 3, activation="relu", padding="same")(x)
    encoded = layers.MaxPooling2D(2, padding="same")(x)

    # Decoder
    x = layers.Conv2D(128, 3, activation="relu", padding="same")(encoded)
    x = layers.UpSampling2D(2)(x)
    x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
    x = layers.UpSampling2D(2)(x)
    x = layers.Conv2D(32, 3, activation="relu", padding="same")(x)
    x = layers.UpSampling2D(2)(x)
    outputs = layers.Conv2D(1, 3, activation="sigmoid", padding="same")(x)

    model = models.Model(inputs, outputs, name="NanoVisionNetX_Reconstructor")
    model.compile(optimizer="adam", loss="mse")
    return model

model = build_autoencoder(input_shape=(*TARGET_SIZE, 1))
model.summary()


In [ ]:
# Train
EPOCHS = 8
BATCH_SIZE = 16

history = model.fit(
    x_train_noisy,
    x_train,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)


In [ ]:
# Evaluate reconstruction quality
reconstructed = model.predict(x_test_noisy, verbose=0)

subset = min(300, len(x_test))
psnr_scores, ssim_scores = [], []
for i in range(subset):
    gt = x_test[i, ..., 0]
    rec = reconstructed[i, ..., 0]
    psnr_scores.append(psnr(gt, rec, data_range=1.0))
    ssim_scores.append(ssim(gt, rec, data_range=1.0))

print(f"Average PSNR ({subset} samples): {np.mean(psnr_scores):.2f} dB")
print(f"Average SSIM ({subset} samples): {np.mean(ssim_scores):.4f}")

# Visual comparison
num_images = min(6, len(x_test))
idx = np.random.choice(len(x_test), size=num_images, replace=False)

plt.figure(figsize=(14, 7))
for i, j in enumerate(idx):
    plt.subplot(3, num_images, i + 1)
    plt.imshow(x_test_noisy[j].squeeze(), cmap="gray")
    plt.title("Noisy")
    plt.axis("off")

    plt.subplot(3, num_images, i + 1 + num_images)
    plt.imshow(reconstructed[j].squeeze(), cmap="gray")
    plt.title("Reconstructed")
    plt.axis("off")

    plt.subplot(3, num_images, i + 1 + 2 * num_images)
    plt.imshow(x_test[j].squeeze(), cmap="gray")
    plt.title("Ground Truth")
    plt.axis("off")

plt.suptitle("NanoVision Net X Reconstruction — nu_data dataset", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Save artifacts
os.makedirs("reconstruction_outputs", exist_ok=True)
model.save("reconstruction_outputs/nanovision_reconstructor_nu_data.keras")

sample_idx = 0
plt.imsave(
    "reconstruction_outputs/sample_reconstruction_nu_data.png",
    reconstructed[sample_idx].squeeze(),
    cmap="gray"
)

print("Saved:")
print("- reconstruction_outputs/nanovision_reconstructor_nu_data.keras")
print("- reconstruction_outputs/sample_reconstruction_nu_data.png")
